# Proposed Model — Gated Fusion (TabTransformer + ModernBERT + DOM)
**Mục tiêu:** 12 URL features + ModernBERT (HTML text) + DOM (64-dim) → Gated Fusion → binary classification
**Dataset:** `mendeley-phishing-2021` → `index.csv` + `html/`
**Thời gian:** ~2h trên GPU T4 (5-fold, 50k samples, pre-tokenized)

In [ ]:
import os, json, re, math, warnings, zipfile, gc, pickle
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
)
from bs4 import BeautifulSoup, Comment
from tqdm.notebook import tqdm

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════
# CONSTANTS
# ══════════════════════════════════════════════
SEED          = 42
N_FOLDS       = 5
EP            = 8
BS            = 16
ACC           = 2
LR_TAB        = 5e-5
LR_BERT       = 1e-5
PATIENCE      = 3
MAX_GRAD_NORM = 0.5
MAX_TEXT_LEN  = 128
DOM_DIM       = 64
BERT_DIM      = 768
TAB_OUT_DIM   = 128
HTML_DIM      = BERT_DIM + DOM_DIM
FUSION_DIM    = 960
FREEZE_LAYERS = 8
SAMPLE_SIZE   = 50000
MODEL_NAME    = 'answerdotai/ModernBERT-base'

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | GPUs: {torch.cuda.device_count()}')

In [ ]:
# ══════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════
KAGGLE_INPUT = Path('/kaggle/input')
PROJECT      = Path('..')
OUT_DIR      = Path('/kaggle/working') if KAGGLE_INPUT.exists() else PROJECT / 'data'
MODEL_DIR    = OUT_DIR / 'models'
FIG_DIR      = OUT_DIR / 'figures'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect Mendeley index.csv
mendeley_idx = None
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.rglob('index.csv'):
        mendeley_idx = p
        print(f'Found index.csv: {mendeley_idx}')
        break
if mendeley_idx is None:
    mendeley_idx = PROJECT / 'data' / 'raw' / 'mendeley' / 'index.csv'

# Auto-detect HTML directory
mendeley_html = mendeley_idx.parent / 'html'
html_zip_path = mendeley_idx.parent / 'html.zip'

if html_zip_path.exists():
    print('Extracting html.zip...')
    with zipfile.ZipFile(html_zip_path, 'r') as zf:
        zf.extractall(str(OUT_DIR))
    mendeley_html = OUT_DIR / 'html'

# Fix nested html/html/ structure
nested = mendeley_html / 'html'
if nested.exists() and nested.is_dir():
    mendeley_html = nested
    print(f'Nested structure detected -> {mendeley_html}')

html_count = len(list(mendeley_html.rglob('*.html'))) if mendeley_html.exists() else 0
print(f'HTML directory : {mendeley_html}')
print(f'HTML files     : {html_count:,}')
if html_count == 0:
    raise RuntimeError(f'No HTML files found in {mendeley_html}. Check Kaggle dataset.')

In [ ]:
# ══════════════════════════════════════════════
# FEATURE EXTRACTION
# ══════════════════════════════════════════════
SUSPICIOUS_KEYWORDS = [
    'login','secure','verify','account','update','banking','confirm',
    'signin','password','reset','authenticate','paypal','webscr','free','bonus'
]
COMMON_TLDS = {
    'com','org','net','gov','edu','mil','io','co','uk','au','de',
    'jp','fr','ca','ru','cn','in','br','pl','html','php','asp','jsp'
}
URL_FEATURE_KEYS = [
    'url_length','domain_length','path_length','entropy',
    'special_char_ratio','digit_ratio','subdomain_count','has_https',
    'has_ip_address','suspicious_keywords','url_depth','tld_in_path'
]
TABULAR_DIM = len(URL_FEATURE_KEYS)

IP_RE = re.compile(
    r'^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}'
    r'(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$'
)

def shannon_entropy(text):
    if not text: return 0.0
    l = len(text)
    return round(-sum(
        (text.count(c)/l) * math.log2(text.count(c)/l)
        for c in set(text) if text.count(c) > 0
    ), 4)

def extract_url_features(url):
    try:
        parsed = urlparse(str(url).strip())
        domain = (parsed.netloc or parsed.hostname or '').split(':')[0]
        path   = parsed.path or ''
        fu     = str(url).strip()
        parts  = domain.split('.')
        tc     = max(len(fu), 1)
        return {
            'url_length':          len(fu),
            'domain_length':       len(domain),
            'path_length':         len(path),
            'entropy':             shannon_entropy(fu),
            'special_char_ratio':  round(sum(1 for c in fu if c in '@-_?.&=%+#~!') / tc, 4),
            'digit_ratio':         round(sum(1 for c in fu if c.isdigit()) / tc, 4),
            'subdomain_count':     max(0, len(parts) - 2) if len(parts) >= 2 else 0,
            'has_https':           1 if parsed.scheme == 'https' else 0,
            'has_ip_address':      1 if IP_RE.match(domain) else 0,
            'suspicious_keywords': sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in fu.lower()),
            'url_depth':           len([s for s in path.split('/') if s]),
            'tld_in_path':         1 if any(f'.{t}' in path.lower() for t in COMMON_TLDS) else 0,
        }
    except Exception:
        return {k: 0.0 for k in URL_FEATURE_KEYS}

In [ ]:
EVENT_HANDLERS = {
    'onclick','ondblclick','onmousedown','onmouseup','onmouseover','onmouseout',
    'onmousemove','onkeydown','onkeypress','onkeyup','onsubmit','onreset',
    'onfocus','onblur','onload','onunload','onchange','onselect','onscroll',
    'onresize','onerror','onabort','oncontextmenu','oninput','oninvalid',
    'ontouchstart','ontouchend','ontouchmove'
}
VOID_TAGS = {
    'area','base','br','col','embed','hr','img','input',
    'link','meta','param','source','track','wbr'
}
SUSPICIOUS_JS = [
    r'\beval\s*\(', r'\bdocument\.write\s*\(',
    r'\bwindow\.location\b', r'\bdocument\.location\b', r'\btop\.location\b'
]

def extract_dom_64(soup, base_url=''):
    try:
        bd = (urlparse(base_url).hostname or '').lower()

        def is_ext(h):
            if not h or h.startswith(('#', 'javascript:')): return False
            try:
                p = urlparse(h)
                return bool(p.hostname and bd and p.hostname.lower() != bd)
            except Exception:
                return False

        scripts  = soup.find_all('script')
        inputs   = soup.find_all('input')
        anchors  = soup.find_all('a', href=True)
        imgs     = soup.find_all('img', src=True)
        all_tags = soup.find_all(True)
        tl       = len(anchors)
        el       = sum(1 for a in anchors if is_ext(a['href']))
        timg     = len(imgs)
        eimg     = sum(1 for im in imgs if is_ext(im['src']))

        hc = sum(
            1 for t in all_tags
            if ('display:none' in t.get('style','').replace(' ','') or
                'visibility:hidden' in t.get('style','').replace(' ','') or
                t.get('hidden') is not None or t.get('type') == 'hidden')
        )
        mr = int(any(
            m.get('http-equiv','').lower() == 'refresh'
            for m in soup.find_all('meta', attrs={'http-equiv': True})
        ))
        ij   = ' '.join(s.string for s in scripts if s.string)
        sjc  = sum(len(re.findall(p, ij, re.I)) for p in SUSPICIOUS_JS)
        ec   = len(re.findall(r'\beval\s*\(', ij))
        dwc  = len(re.findall(r'\bdocument\.write\s*\(', ij))
        fav  = soup.find('link', rel=lambda v: v and 'icon' in ' '.join(v).lower())
        fe   = 1 if (fav and fav.get('href') and is_ext(fav['href'])) else 0
        attr_c  = sum(len(t.attrs) for t in all_tags)
        event_c = sum(1 for t in all_tags for a in t.attrs if a.lower() in EVENT_HANDLERS)
        avg_a   = round(attr_c / max(len(all_tags), 1), 4)
        elc     = sum(1 for a in anchors if not a.get('href') or a['href'].strip() in ('#',''))

        js_syn = [min(ij.count(x), 9999) for x in ['http','https','.','=','+','[','{','(']]
        js_kw  = [min(ij.count(x), 9999) for x in [
            'function','var ','let ','const ','return',
            'if (','for (','while (','try ','catch ',
            'new ','this.','null','undefined','true','false'
        ]]

        groups = [
            [len(scripts), len(soup.find_all('iframe')), len(soup.find_all('form')),
             len(inputs), sum(1 for i in inputs if i.get('type')=='password'),
             len(soup.find_all('button')), tl],
            [sum(1 for s in scripts if s.get('src') and is_ext(s['src'])),
             round(el/max(tl,1),4), round(eimg/max(timg,1),4), fe, timg, el],
            [hc, mr, sjc, ec, dwc, elc],
            [len(soup.find_all('meta')), len(soup.find_all('div')),
             len(soup.find_all('p')),    len(soup.find_all('table')),
             len(soup.find_all('span')), len(soup.find_all('ul'))],
            [len(soup.find_all('li')),   len(soup.find_all(re.compile(r'^h[1-6]$'))),
             len(soup.find_all('br')),   len(soup.find_all('style')),
             len(soup.find_all('link')),
             len(soup.find_all(['object','embed','applet','video','audio','canvas','svg']))],
            [len(soup.find_all(['nav','header','footer','section','article','aside','main'])),
             sum(1 for t in all_tags if t.name.lower() in VOID_TAGS),
             event_c, min(attr_c, 99999), avg_a],
            js_syn,
            js_kw,
        ]
        vec = np.concatenate([np.array(g, dtype=np.float32) for g in groups])
        vec = np.clip(vec, 0, 1e6)
        vec = np.nan_to_num(vec, nan=0.0)
        if len(vec) < DOM_DIM:
            vec = np.pad(vec, (0, DOM_DIM - len(vec)))
        return vec[:DOM_DIM]
    except Exception:
        return np.zeros(DOM_DIM, dtype=np.float32)

def extract_clean_text(soup, max_c=4096):
    try:
        soup_copy = BeautifulSoup(str(soup), 'html.parser')
        for t in soup_copy(['script','style','noscript']): t.decompose()
        for c in soup_copy.find_all(string=lambda s: isinstance(s, Comment)): c.extract()
        return re.sub(r'\s+', ' ', soup_copy.get_text(separator=' ', strip=True)).strip()[:max_c]
    except Exception:
        return ''

In [ ]:
# ══════════════════════════════════════════════
# LOAD, SAMPLE & PREPROCESS
# ══════════════════════════════════════════════
df = pd.read_csv(mendeley_idx, encoding='utf-8')
print(f'\nFull dataset : {len(df):,} | Phishing: {(df["result"]==1).sum():,} | Genuine: {(df["result"]==0).sum():,}')

n_full = len(df); n_full_ph = int((df['result']==1).sum()); n_full_be = int((df['result']==0).sum())

n_each = SAMPLE_SIZE // 2
df_p   = df[df['result']==1].sample(n=min(n_each, (df['result']==1).sum()), random_state=SEED)
df_g   = df[df['result']==0].sample(n=min(n_each, (df['result']==0).sum()), random_state=SEED)
df     = pd.concat([df_p, df_g]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Sampled      : {len(df):,} | Phishing: {(df["result"]==1).sum():,} | Genuine: {(df["result"]==0).sum():,}')

full_data     = []
found_count   = 0
missing_count = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Preprocessing HTML'):
    url   = str(row['url']).strip()
    fname = str(row['website']).strip()
    label = int(row['result'])
    uv    = [extract_url_features(url)[k] for k in URL_FEATURE_KEYS]
    subdir= 'genuine' if label == 0 else 'phishing'
    hp    = mendeley_html / subdir / fname

    if hp.exists():
        try:
            html = hp.read_text(encoding='utf-8', errors='replace')
            soup = BeautifulSoup(html, 'html.parser')
            dv   = extract_dom_64(soup, base_url=url)
            ct   = extract_clean_text(soup)
            found_count += 1
        except Exception:
            dv, ct = np.zeros(DOM_DIM, dtype=np.float32), ''
            missing_count += 1
    else:
        dv, ct = np.zeros(DOM_DIM, dtype=np.float32), ''
        missing_count += 1

    full_data.append({
        'url_features': uv,
        'dom_features': dv.tolist(),
        'clean_text':   ct,
        'label':        label,
    })

print(f'HTML found   : {found_count:,} | Missing: {missing_count:,}')
if missing_count == len(full_data):
    raise RuntimeError('All HTML missing - check mendeley_html path!')

# Stratified 80/20 train/test split — test NEVER used in CV (same logic as baseline2)
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=df['result'].values
)
train_idx = np.array(train_idx, dtype=np.int64)
test_idx  = np.array(test_idx,  dtype=np.int64)
print(f'Train: {len(train_idx):,} | Test (held-out): {len(test_idx):,}')
with open(OUT_DIR / 'proposed_splits.json', 'w') as f:
    json.dump({'train_indices': train_idx.tolist(), 'test_indices': test_idx.tolist()}, f, indent=2)
print(f'Saved: {OUT_DIR / "proposed_splits.json"}')

# ── Dataset statistics (for thesis figures) ──
dataset_stats = {
    'dataset': 'Mendeley 2021 (HTML)',
    'n_full': n_full,
    'n_full_phishing': n_full_ph,
    'n_full_benign': n_full_be,
    'n_samples': int(len(df)),
    'n_phishing': int((df['result']==1).sum()),
    'n_benign': int((df['result']==0).sum()),
    'phishing_ratio': round(float((df['result']==1).mean()), 6),
    'html_found': found_count,
    'html_missing': missing_count,
    'n_features': len(URL_FEATURE_KEYS),
    'feature_keys': URL_FEATURE_KEYS,
}
with open(OUT_DIR / 'dataset_stats_proposed.json', 'w') as f:
    json.dump(dataset_stats, f, indent=2)
print(f'Stats saved to {OUT_DIR / "dataset_stats_proposed.json"}')

In [ ]:
# Pre-tokenize 1 lần, cache vao RAM
print('\nPre-tokenizing (one-time)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
TOK_BS    = 512
ids_list, mask_list = [], []

for i in tqdm(range(0, len(full_data), TOK_BS), desc='Tokenizing'):
    batch = [d['clean_text'] for d in full_data[i:i+TOK_BS]]
    enc   = tokenizer(
        batch, padding='max_length', truncation=True,
        max_length=MAX_TEXT_LEN, return_tensors='pt'
    )
    ids_list.append(enc['input_ids'])
    mask_list.append(enc['attention_mask'])

all_ids  = torch.cat(ids_list,  dim=0)
all_mask = torch.cat(mask_list, dim=0)

for i, d in enumerate(full_data):
    d['input_ids']      = all_ids[i]
    d['attention_mask'] = all_mask[i]

labels = [d['label'] for d in full_data]
print('Tokenization & cache done.')

In [ ]:
# ══════════════════════════════════════════════
# MODEL ARCHITECTURE
# ══════════════════════════════════════════════
class FeatureEmbedding(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.e = nn.Linear(1, d)
    def forward(self, x):
        return self.e(x.unsqueeze(-1))

class TabTransformer(nn.Module):
    def __init__(self, nf=TABULAR_DIM, ed=32, nh=4, hd=128, od=TAB_OUT_DIM, dp=0.1):
        super().__init__()
        self.embs = nn.ModuleList([FeatureEmbedding(ed) for _ in range(nf)])
        self.attn = nn.MultiheadAttention(ed, nh, batch_first=True, dropout=dp)
        self.n1   = nn.LayerNorm(ed)
        self.n2   = nn.LayerNorm(ed)
        self.ff   = nn.Sequential(
            nn.Linear(ed, hd), nn.GELU(), nn.Dropout(dp),
            nn.Linear(hd, ed), nn.Dropout(dp)
        )
        self.proj = nn.Sequential(
            nn.Linear(ed * nf, od),
            nn.LayerNorm(od)
        )
    def forward(self, x):
        h    = torch.stack([e(x[:, i]) for i, e in enumerate(self.embs)], dim=1)
        a, _ = self.attn(h, h, h)
        h    = self.n1(h + a)
        h    = self.n2(h + self.ff(h))
        return self.proj(h.reshape(h.size(0), -1))

class ModernBERTBranch(nn.Module):
    def __init__(self, freeze_layers=FREEZE_LAYERS, dp=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        enc    = (getattr(self.bert, 'encoder', None) or
                  getattr(self.bert, 'model',   None) or self.bert)
        layers = getattr(enc, 'layer', None) or getattr(enc, 'layers', None) or []
        for i, layer in enumerate(layers):
            if i < freeze_layers:
                for p in layer.parameters():
                    p.requires_grad = False
        self.proj = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, BERT_DIM),
            nn.GELU(),
            nn.LayerNorm(BERT_DIM),
            nn.Dropout(dp)
        )
    def forward(self, ids, mask):
        o = self.bert(input_ids=ids, attention_mask=mask)
        return self.proj(o.last_hidden_state[:, 0])

class GatedFusion(nn.Module):
    def __init__(self, url_dim=TAB_OUT_DIM, html_dim=HTML_DIM, fused_dim=FUSION_DIM):
        super().__init__()
        self.fused_dim = fused_dim
        self.url_proj  = nn.Linear(url_dim, fused_dim)
        self.gate      = nn.Sequential(
            nn.Linear(fused_dim + html_dim, fused_dim),
            nn.Sigmoid()
        )
        self.out = nn.Sequential(
            nn.Linear(fused_dim, fused_dim),
            nn.LayerNorm(fused_dim)
        )
    def forward(self, v_url, v_html):
        v_up     = self.url_proj(v_url)
        gate     = self.gate(torch.cat([v_up, v_html], dim=-1))
        v_html_p = F.pad(v_html, (0, self.fused_dim - v_html.size(-1)))
        return self.out(gate * v_up + (1 - gate) * v_html_p)

class PhishingDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.tab    = TabTransformer()
        self.bert   = ModernBERTBranch()
        self.dom    = nn.Sequential(
            nn.Linear(DOM_DIM, DOM_DIM),
            nn.ReLU(),
            nn.LayerNorm(DOM_DIM)
        )
        self.fusion = GatedFusion()
        self.cls    = nn.Sequential(
            nn.Linear(FUSION_DIM, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64),         nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, tab, ids, mask, dom):
        v_url  = self.tab(tab)
        v_bert = self.bert(ids, mask)
        v_dom  = self.dom(dom)
        v_html = torch.cat([v_bert, v_dom], dim=-1)
        v_fuse = self.fusion(v_url, v_html)
        return self.cls(v_fuse)

In [ ]:
# ══════════════════════════════════════════════
# DATASET & UTILITIES
# ══════════════════════════════════════════════
class CachedDataset(Dataset):
    def __init__(self, records, indices, url_mean, url_std, dom_mean, dom_std):
        self.data = [records[i] for i in indices]
        self.url_mean = url_mean
        self.url_std = url_std
        self.dom_mean = dom_mean
        self.dom_std = dom_std
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        r = self.data[i]
        norm_url = (np.array(r['url_features'], dtype=np.float32) - self.url_mean) / self.url_std
        norm_dom = (np.array(r['dom_features'], dtype=np.float32) - self.dom_mean) / self.dom_std
        return (
            torch.tensor(norm_url, dtype=torch.float32),
            r['input_ids'],
            r['attention_mask'],
            torch.tensor(norm_dom, dtype=torch.float32),
            torch.tensor(r['label'], dtype=torch.float32),
        )

def collate_fn(batch):
    tab, ids, mask, dom, lbl = zip(*batch)
    return (
        torch.stack(tab).to(DEVICE),
        torch.stack(ids).to(DEVICE),
        torch.stack(mask).to(DEVICE),
        torch.stack(dom).to(DEVICE),
        torch.stack(lbl).unsqueeze(1).to(DEVICE),
    )

def compute_metrics(labs, preds):
    preds = np.nan_to_num(preds, nan=0.5, posinf=1.0, neginf=0.0)
    preds = np.clip(preds, 0.0, 1.0)
    pb    = (preds >= 0.5).astype(int)
    fpr_val = 0.0
    if len(np.unique(labs)) > 1:
        cm_     = confusion_matrix(labs, pb)
        tn, fp  = cm_[0,0], cm_[0,1]
        fpr_val = round(fp / max(tn + fp, 1), 4)
    return {
        'accuracy':  accuracy_score(labs, pb),
        'precision': precision_score(labs, pb, zero_division=0),
        'recall':    recall_score(labs,    pb, zero_division=0),
        'f1':        f1_score(labs,        pb, zero_division=0),
        'auc':       roc_auc_score(labs, preds) if len(np.unique(labs)) > 1 else 0.0,
        'fpr':       fpr_val,
    }

In [ ]:
# ══════════════════════════════════════════════
# TRAINING & EVALUATION FUNCTIONS
# ══════════════════════════════════════════════
def train_epoch(model, loader, opt, crit, scaler):
    model.train()
    total_loss     = 0.0
    pending_update = False
    opt.zero_grad()
    n = len(loader)

    for step, (tab, ids, mask, dom, lbl) in enumerate(loader):
        with torch.amp.autocast(device_type='cuda', enabled=DEVICE.type=='cuda'):
            logits = model(tab, ids, mask, dom)
            loss   = crit(logits, lbl)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  [WARN] NaN/Inf loss at step {step}, skipping')
            opt.zero_grad()
            pending_update = False
            continue

        scaler.scale(loss).backward()
        pending_update = True
        total_loss += loss.item() * tab.size(0)

        if (step + 1) % ACC == 0:
            scaler.unscale_(opt)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=MAX_GRAD_NORM
            )
            if not torch.isnan(grad_norm):
                scaler.step(opt)
            scaler.update()
            opt.zero_grad()
            pending_update = False

        if (step + 1) % 300 == 0:
            print(f'    step {step+1}/{n} | loss {loss.item():.4f}', flush=True)

    if pending_update:
        scaler.unscale_(opt)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=MAX_GRAD_NORM
        )
        if not torch.isnan(grad_norm):
            scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    return total_loss / max(len(loader.dataset), 1)

def evaluate(model, loader, crit):
    model.eval()
    total_loss = 0.0
    preds, labs = [], []
    with torch.no_grad():
        for tab, ids, mask, dom, lbl in loader:
            with torch.amp.autocast(device_type='cuda', enabled=DEVICE.type=='cuda'):
                logits = model(tab, ids, mask, dom)
                loss   = crit(logits, lbl)
            total_loss += loss.item() * tab.size(0)
            p = torch.sigmoid(logits).cpu().numpy().flatten()
            preds.extend(p)
            labs.extend(lbl.cpu().numpy().flatten())

    preds = np.nan_to_num(np.array(preds), nan=0.5)
    labs  = np.array(labs)
    m     = compute_metrics(labs, preds)
    m['loss'] = total_loss / len(loader.dataset)
    return m, preds, labs

In [ ]:
# ══════════════════════════════════════════════
# N-FOLD CROSS VALIDATION (held-out test)
# ══════════════════════════════════════════════
all_labels_arr = np.array([d['label'] for d in full_data], dtype=np.float32)
train_labels_arr = all_labels_arr[train_idx]
test_labels_arr  = all_labels_arr[test_idx]
pos_weight = torch.tensor([(len(train_labels_arr) - train_labels_arr.sum()) / max(train_labels_arr.sum(), 1)], device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f'pos_weight={pos_weight.item():.2f} (neg={int(len(train_labels_arr)-train_labels_arr.sum())}, pos={int(train_labels_arr.sum())})')

skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_metrics, test_metrics = [], []
test_preds = []
folds_meta, history = [], []

for fold, (tr_rel, te_rel) in enumerate(skf.split(train_idx, train_labels_arr)):
    tr_idx = train_idx[tr_rel]
    te_idx = train_idx[te_rel]
    print(f'\n{"="*58}')
    print(f'  FOLD {fold+1}/{N_FOLDS}  |  train={len(tr_idx):,}  val={len(te_idx):,}')
    print(f'{"="*58}')

    tr_url_arr = np.array([full_data[i]['url_features'] for i in tr_idx], dtype=np.float32)
    url_mean, url_std = tr_url_arr.mean(axis=0), tr_url_arr.std(axis=0) + 1e-8
    tr_dom_arr = np.array([full_data[i]['dom_features'] for i in tr_idx], dtype=np.float32)
    dom_mean, dom_std = tr_dom_arr.mean(axis=0), tr_dom_arr.std(axis=0) + 1e-8

    model   = PhishingDetector().to(DEVICE)
    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Params: total={total_p:,} | trainable={train_p:,}')

    tr_ld = DataLoader(
        CachedDataset(full_data, tr_idx, url_mean, url_std, dom_mean, dom_std),
        batch_size=BS, shuffle=True, collate_fn=collate_fn, num_workers=0
    )
    te_ld = DataLoader(
        CachedDataset(full_data, te_idx, url_mean, url_std, dom_mean, dom_std),
        batch_size=BS*2, shuffle=False, collate_fn=collate_fn, num_workers=0
    )

    opt = torch.optim.AdamW([
        {'params': model.tab.parameters(),    'lr': LR_TAB},
        {'params': model.bert.parameters(),   'lr': LR_BERT},
        {'params': model.dom.parameters(),    'lr': LR_TAB},
        {'params': model.fusion.parameters(), 'lr': LR_TAB},
        {'params': model.cls.parameters(),    'lr': LR_TAB},
    ], weight_decay=1e-5, eps=1e-8)

    def lr_lambda(ep):
        return 0.3 if ep == 1 else 1.0
    warmup = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP-1, eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.ChainedScheduler([warmup, cosine])

    scaler = torch.amp.GradScaler(enabled=DEVICE.type=='cuda')

    best_f1, patience_count = 0.0, 0
    best_ckpt = MODEL_DIR / f'proposed_fold{fold+1}_best.pt'
    hist = []

    for ep in range(1, EP + 1):
        tl      = train_epoch(model, tr_ld, opt, crit, scaler)
        m, _, _ = evaluate(model, te_ld, crit)
        scheduler.step()

        hist.append({'epoch': ep, 'train_loss': round(float(tl), 5),
                     'val_auc': round(float(m['auc']), 5), 'val_f1': round(float(m['f1']), 5)})

        flag = ''
        if m['f1'] > best_f1:
            best_f1        = m['f1']
            patience_count = 0
            torch.save(model.state_dict(), best_ckpt)
            flag = '   <- best'
        else:
            patience_count += 1

        print(
            f'  Ep {ep:2d}/{EP} | '
            f'Loss {tl:.4f} | Val {m["loss"]:.4f} | '
            f'AUC {m["auc"]:.4f} | F1 {m["f1"]:.4f} | '
            f'Acc {m["accuracy"]:.4f} | FPR {m["fpr"]:.4f}'
            f'{flag}'
        )

        if patience_count >= PATIENCE:
            print(f'  Early stopping at epoch {ep}')
            break

    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    fm, _, _ = evaluate(model, te_ld, crit)
    fm['fold']  = fold + 1
    all_metrics.append(fm)
    history.append({'fold': fold + 1, 'epochs': hist})

    print(f'\n  Fold {fold+1} -> CV Acc={fm["accuracy"]:.4f} | AUC={fm["auc"]:.4f} | F1={fm["f1"]:.4f}')

    folds_meta.append({
        'fold': int(fold + 1),
        'url_mean': url_mean.tolist(),
        'url_std': url_std.tolist(),
        'dom_mean': dom_mean.tolist(),
        'dom_std': dom_std.tolist(),
    })

    # Held-out test evaluation using this fold's train scaler (test never in CV)
    test_ld = DataLoader(
        CachedDataset(full_data, test_idx, url_mean, url_std, dom_mean, dom_std),
        batch_size=BS*2, shuffle=False, collate_fn=collate_fn, num_workers=0
    )
    tm, tp, _ = evaluate(model, test_ld, crit)
    tm['fold']  = fold + 1
    test_metrics.append(tm)
    test_preds.append(tp)
    print(f'  Fold {fold+1} -> Test Acc={tm["accuracy"]:.4f} | AUC={tm["auc"]:.4f} | F1={tm["f1"]:.4f}')

    del model, opt, scheduler, scaler, tr_ld, te_ld, test_ld
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Save artifacts for evaluation/figures ──
with open(MODEL_DIR / 'proposed_folds.json', 'w') as f:
    json.dump({'n_folds': N_FOLDS, 'folds': folds_meta}, f, indent=2)
with open(OUT_DIR / 'training_logs_proposed.json', 'w') as f:
    json.dump(history, f, indent=2)
test_preds_mean = np.mean(np.stack(test_preds), axis=0)
tp_obj = np.empty(N_FOLDS, dtype=object)
for f in range(N_FOLDS):
    tp_obj[f] = test_preds[f]
np.savez(OUT_DIR / 'predictions_proposed.npz',
         test_preds_mean=test_preds_mean, test_labels=test_labels_arr, test_preds=tp_obj)
print(f'Artifacts saved: proposed_folds.json, training_logs_proposed.json, predictions_proposed.npz')

metric_keys = ['accuracy','precision','recall','f1','auc','fpr']
avg = {k: np.mean([m[k] for m in all_metrics]) for k in metric_keys}
std = {k: np.std( [m[k] for m in all_metrics]) for k in metric_keys}
test_avg = {k: np.mean([m[k] for m in test_metrics]) for k in metric_keys}
test_std = {k: np.std( [m[k] for m in test_metrics]) for k in metric_keys}

print(f'\n{"="*58}\n  {N_FOLDS}-FOLD CV SUMMARY (train portion)\n{"="*58}')
for k in metric_keys:
    print(f'  {k.upper():10s}: {avg[k]:.4f} +/- {std[k]:.4f}')
print(f'\n{"="*58}\n  HELD-OUT TEST (never in CV)\n{"="*58}')
for k in metric_keys:
    print(f'  {k.upper():10s}: {test_avg[k]:.4f} +/- {test_std[k]:.4f}')

In [ ]:
# ══════════════════════════════════════════════
# SUMMARY & VISUALIZATION (held-out test)
# ══════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#a6cee3','#1f78b4','#b2df8a','#33a02c','#fb9a99']
a = axes.flatten()

cm_mat = confusion_matrix(test_labels_arr, (test_preds_mean >= 0.5).astype(int))
sns.heatmap(cm_mat, annot=True, fmt='d', cmap='Greens', ax=a[0],
            xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
a[0].set_title('Proposed — Confusion Matrix (held-out test)'); a[0].set_ylabel('True'); a[0].set_xlabel('Predicted')

for f in range(N_FOLDS):
    fpr_c, tpr_c, _ = roc_curve(test_labels_arr, test_preds[f])
    a[1].plot(fpr_c, tpr_c, color=colors[f], lw=1.5, alpha=0.7,
              label=f'Fold {f+1} (AUC={roc_auc_score(test_labels_arr, test_preds[f]):.4f})')
fpr_a, tpr_a, _ = roc_curve(test_labels_arr, test_preds_mean)
a[1].plot(fpr_a, tpr_a, 'k--', lw=2.5, label=f'Ensemble (AUC={roc_auc_score(test_labels_arr, test_preds_mean):.4f})')
a[1].plot([0,1],[0,1], 'gray', lw=1, alpha=0.5)
a[1].set_title('ROC Curves (held-out test)'); a[1].set_xlabel('FPR'); a[1].set_ylabel('TPR')
a[1].legend(fontsize=8, loc='lower right')

show_keys = ['accuracy','precision','recall','f1','auc']
x = np.arange(len(show_keys)); means = [test_avg[m] for m in show_keys]; stdevs = [test_std[m] for m in show_keys]
a[2].bar(x, means, yerr=stdevs, capsize=5, color='#33a02c', alpha=0.8)
a[2].set_xticks(x); a[2].set_xticklabels([k.capitalize() for k in show_keys])
a[2].set_ylim(0, 1); a[2].set_title('Metrics — Held-out Test (Mean+-Std)')
for i, (m, s) in enumerate(zip(means, stdevs)):
    a[2].text(i, m + s + 0.02, f'{m:.3f}+-{s:.3f}', ha='center', fontsize=8)

# Training curves — mean across folds
av = []
max_ep = max(len(h['epochs']) for h in history)
for ep in range(1, max_ep + 1):
    rows = [h['epochs'][ep-1] for h in history if len(h['epochs']) >= ep]
    av.append({'epoch': ep, **{k: float(np.mean([r[k] for r in rows])) for k in ['train_loss','val_auc','val_f1']}})
a[3].plot([r['epoch'] for r in av], [r['train_loss'] for r in av], 'o-', color='#d62728', lw=1.5, label='Train loss')
a[3].set_xlabel('Epoch'); a[3].set_ylabel('Loss', color='#d62728')
a[3].tick_params(axis='y', labelcolor='#d62728')
a3b = a[3].twinx()
a3b.plot([r['epoch'] for r in av], [r['val_auc'] for r in av], 's-', color='#1f78b4', lw=1.5, label='Val AUC')
a3b.plot([r['epoch'] for r in av], [r['val_f1'] for r in av], 'd-', color='#33a02c', lw=1.5, label='Val F1')
a3b.set_ylim(0, 1); a3b.set_ylabel('Score')
a3b.legend(fontsize=8, loc='lower left')
a[3].set_title('Training Curves (mean across folds)')

# Data distribution (full + sampled)
w = 0.38; xs = np.arange(2)
a[4].bar(xs - w/2, [dataset_stats['n_full_benign'], dataset_stats['n_full_phishing']], w,
         label='Full', color=['#2ca02c','#d62728'], alpha=0.45)
a[4].bar(xs + w/2, [dataset_stats['n_benign'], dataset_stats['n_phishing']], w,
         label='Sampled', color=['#2ca02c','#d62728'], alpha=0.9)
a[4].set_xticks(xs); a[4].set_xticklabels(['Benign','Phishing'])
a[4].set_ylabel('Samples')
a[4].set_title(f"Mendeley Distribution (full={dataset_stats['n_full']:,}, sampled={dataset_stats['n_samples']:,})")
a[4].legend(fontsize=8)

a[5].axis('off')
a[5].text(0.02, 0.95, 'Proposed — TabTransformer + ModernBERT + GatedFusion', fontsize=13, weight='bold', va='top')
a[5].text(0.02, 0.75, f"Held-out test (5-fold ensemble):\nAcc={test_avg['accuracy']:.4f}+-{test_std['accuracy']:.4f}\n"
                      f"AUC={test_avg['auc']:.4f}+-{test_std['auc']:.4f}\n"
                      f"F1={test_avg['f1']:.4f}+-{test_std['f1']:.4f}\n"
                      f"Precision={test_avg['precision']:.4f}+-{test_std['precision']:.4f}\n"
                      f"Recall={test_avg['recall']:.4f}+-{test_std['recall']:.4f}", fontsize=11, va='top')

plt.tight_layout(); plt.savefig(FIG_DIR / 'proposed_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "proposed_summary.png"}')

In [ ]:
# ══════════════════════════════════════════════
# SAVE RESULTS JSON (top-level = held-out test, cv_* = CV)
# ══════════════════════════════════════════════
results = {
    'model': 'TabTransformer + ModernBERT + GatedFusion',
    'sample_size': SAMPLE_SIZE,
    'n_folds': N_FOLDS
}
for k in metric_keys:
    results[k] = round(float(test_avg[k]), 6)
    results[k + '_std'] = round(float(test_std[k]), 6)
    results['cv_' + k] = round(float(avg[k]), 6)
    results['cv_' + k + '_std'] = round(float(std[k]), 6)

with open(MODEL_DIR / 'evaluation_proposed.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved to evaluation_proposed.json (top-level = held-out test)')

---
### Download từ Output tab:
- `figures/proposed_summary.png` (báo cáo)
- `data/models/evaluation_proposed.json`
- `data/models/proposed_folds.json`
- `proposed_splits.json`
- `training_logs_proposed.json`
- `predictions_proposed.npz`
- `dataset_stats_proposed.json`
- `data/models/proposed_fold*_best.pt` (optional)

Sau đó chạy **Compare** → `kaggle_compare.ipynb`